# Demo 04 — PDF to Chroma, end to end

A single walk from a **real PDF** to a **queryable Chroma-based RAG**.

Pipeline: **extract → clean → chunk → enrich → dedup/version/access → embed → index with metadata → retrieve**.

In [1]:
# Dependencies:
# !pip install pymupdf4llm ftfy unstructured langchain-text-splitters sentence-transformers chromadb dspy
# No API key needed — everything runs locally on CPU.

In [2]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")   # this Mac's torch build segfaults on MPS
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")

import warnings, logging, re, hashlib
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

import numpy as np
import ftfy
import pymupdf4llm
from unstructured.cleaners.core import clean_extra_whitespace
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

PDF_PATH = "demo04_pdf_preprocessing_handbook.pdf"   # shared with demo04_pdf_preprocessing
HEADER = "Bright Valley College — Student Handbook"
EMBED_MODEL = "all-MiniLM-L6-v2"                                # same embedder as demo04_vector_stores

embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")

def embed(texts):
    return np.asarray(embed_model.encode(list(texts), normalize_embeddings=True,
                                         convert_to_numpy=True), dtype=np.float32)

def embed_query(text):
    return embed([text])[0]

print("ready")

ready


## 1 — Extract: structure-aware Markdown, per page  (§6.2)

`pymupdf4llm.to_markdown(path, page_chunks=True)` detects the two-column layout, recovers headings
and the ruled table as Markdown, joins the hyphenated line break, and returns one dict **per PDF
page** with real `page_number` metadata — exactly the extraction layer from
`demo04_pdf_preprocessing`.

In [3]:
pages = pymupdf4llm.to_markdown(PDF_PATH, page_chunks=True)
print(f"{len(pages)} pages extracted from {PDF_PATH}")
print(pages[1]["text"][:300], "...")

3 pages extracted from demo04_pdf_preprocessing_handbook.pdf
**Bright Valley College — Student Handbook** 

# **Campus Policies** 

### **Vacation Policy (2025)** 

### **Compensation (Staff)** 

Students are entitled to 24 vacation days per academic year. Up to 5 unused days carry over to the next year. Requests must be submitted at least two weeks in advanc ...


## 2 — Clean: fix Unicode, strip page furniture  (§6.3)

`ftfy.fix_text` handles Unicode/mojibake normalization and `unstructured`'s `clean_extra_whitespace`
collapses whitespace — the same honest split as `demo04_preprocessing_tools`. Page-furniture removal
(the running header, the `Confidential … Page N` footer) stays custom: no generic cleaner knows
*this* line is boilerplate rather than content.

In [4]:
def clean_page(text):
    text = ftfy.fix_text(text)                                        # [ftfy] unicode / mojibake / NFC
    kept = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s.strip("* ") == HEADER or s.startswith("Confidential — Internal Use Only"):
            continue                                                  # [custom] drop page furniture
        kept.append(clean_extra_whitespace(line))                     # [unstructured] collapse whitespace
    return "\n".join(kept)

cleaned_pages = [{"page": p["metadata"]["page_number"], "text": clean_page(p["text"])} for p in pages]
print(cleaned_pages[1]["text"])

# **Campus Policies**
### **Vacation Policy (2025)**
### **Compensation (Staff)**
Students are entitled to 24 vacation days per academic year. Up to 5 unused days carry over to the next year. Requests must be submitted at least two weeks in advance through the portal.
Teaching assistants are paid 55 NIS per hour for approved hours. Senior lecturers earn a base salary of 28,000 NIS per month, reviewed annually by the department chair.
## **Library Fees**
The following fines apply to overdue library materials.
|**Item type**|**Fine per day**|**Maximum fine**|
|---|---|---|
|Book|2 NIS|40 NIS|
|Laptop|20 NIS|300 NIS|
|Reference|5 NIS|90 NIS|


## 3 — Chunk: split on structure, keep the page number  (§6.4)

`MarkdownHeaderTextSplitter` splits each page on its headings; `RecursiveCharacterTextSplitter`
only re-splits sections that exceed the size cap. Chunking runs **per page** so the real
`page_number` from step 1 rides along next to the heading path — useful for citations
("page 2, Vacation Policy").

In [5]:
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
    strip_headers=False,
)
small_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0)

def crumb(meta):
    return " > ".join(meta[k] for k in ("h1", "h2", "h3") if k in meta)

chunks = []
for cp in cleaned_pages:
    if not cp["text"].strip():
        continue
    sections = header_splitter.split_text(cp["text"])
    for d in small_splitter.split_documents(sections):
        chunks.append({"page": cp["page"], "section": crumb(d.metadata), "text": d.page_content})

for c in chunks:
    print(f"[p{c['page']}] [{c['section']}]  {c['text'][:50]!r}")

[p1] [**Course Policy** > **Assignments**]  '# **Course Policy**  \n## **Assignments**\nAssignmen'
[p1] [**Course Policy** > **Assignments** > **Late submission**]  '### **Late submission**\nAssignments submitted afte'
[p1] [**Course Policy** > **Assignments** > **Grading**]  '### **Grading**\nThe passing grade for every course'
[p2] [**Campus Policies** > **Vacation Policy (2025)**]  '# **Campus Policies**  \n### **Vacation Policy (202'
[p2] [**Campus Policies** > **Compensation (Staff)**]  '### **Compensation (Staff)**\nStudents are entitled'
[p2] [**Campus Policies** > **Library Fees**]  '## **Library Fees**\nThe following fines apply to o'
[p3] [**Campus Map**]  '# **Campus Map**\n_Figure 1. The main library readi'


## 4 — Enrich: version, access, and the section path  (§6.5)

`source`/`doc_type` come from the document name; `version` and `access` are parsed here from the
heading text as a stand-in for what a real document-management system would supply directly. The
section path is folded into `embed_text` so the embedding model sees both the answer **and** its
semantic location.

In [6]:
def enrich(section):
    m = {"source": "student_handbook", "doc_type": "handbook", "access": "public"}
    if (v := re.search(r"\((\d{4})\)", section)):
        m["version"] = int(v.group(1))
    if "Confidential" in section or "Compensation" in section:
        m["access"] = "confidential"
    return m

records = []
for c in chunks:
    records.append({
        "text": c["text"],
        "embed_text": f"Document section: {c['section']}\n{c['text']}" if c["section"] else c["text"],
        "page": c["page"],
        "section": c["section"],
        **enrich(c["section"]),
    })

for r in records:
    print({k: v for k, v in r.items() if k in ("page", "section", "version", "access")})

{'page': 1, 'section': '**Course Policy** > **Assignments**', 'access': 'public'}
{'page': 1, 'section': '**Course Policy** > **Assignments** > **Late submission**', 'access': 'public'}
{'page': 1, 'section': '**Course Policy** > **Assignments** > **Grading**', 'access': 'public'}
{'page': 2, 'section': '**Campus Policies** > **Vacation Policy (2025)**', 'access': 'public', 'version': 2025}
{'page': 2, 'section': '**Campus Policies** > **Compensation (Staff)**', 'access': 'confidential'}
{'page': 2, 'section': '**Campus Policies** > **Library Fees**', 'access': 'public'}
{'page': 3, 'section': '**Campus Map**', 'access': 'public'}


## 5 — Dedup, before indexing  (§6.6)

No library does this for you — it is RAG-specific policy. Exact duplicates are caught by hashing
the raw chunk text; near-duplicates (e.g. two versions of the same policy) by cosine similarity on
the enriched embeddings. This PDF only carries one Vacation Policy version, so both checks come back
empty here — but `version` is already sitting in the metadata from step 4, ready for the day a real
conflict needs resolving.

In [7]:
def sha(t): return hashlib.sha1(t.encode()).hexdigest()[:8]
exact = {}
for r in records:
    exact.setdefault(sha(r["text"]), []).append(r["section"])
print("exact-duplicate groups:", [g for g in exact.values() if len(g) > 1] or "none")

vecs = embed([r["embed_text"] for r in records])
pairs = [(float(vecs[i] @ vecs[j]), records[i], records[j])
         for i in range(len(records)) for j in range(i + 1, len(records)) if vecs[i] @ vecs[j] > 0.80]
print("near-duplicate pairs (cosine > 0.8):",
      [f"{c:.3f} [{a['section']}]~[{b['section']}]" for c, a, b in pairs] or "none")

exact-duplicate groups: none
near-duplicate pairs (cosine > 0.8): none


## 6 — Index into Chroma  (§7.3)

Same three calls as `demo04_vector_stores`: `get_or_create_collection`, `add`, `query`. The chunk
text is the retrievable `document`; `page`, `section`, `access` and (when present) `version` ride
along as filterable metadata.

In [8]:
import chromadb

chroma_client = chromadb.EphemeralClient()
col = chroma_client.get_or_create_collection("pdf_handbook", metadata={"hnsw:space": "cosine"})

ids = [f"chunk_{i:03d}" for i in range(len(records))]
metadatas = [{"page": r["page"], "section": r["section"], "source": r["source"],
              "doc_type": r["doc_type"], "access": r["access"], "version": r.get("version", 0)}
             for r in records]
col.add(ids=ids, documents=[r["text"] for r in records],
        embeddings=vecs.tolist(), metadatas=metadatas)
print("Documents indexed in Chroma collection:", col.count())

Documents indexed in Chroma collection: 7


## 7 — `ChromaRetriever`: a DSPy module over the collection  (§7.8)

The same `ChromaRetriever` from `demo04_vector_stores` — a thin `dspy.Module` wrapping `collection.query`
so the PDF chunk collection can sit behind any DSPy program. `forward` takes an optional
`access_type` (default `"public"`) and folds it straight into Chroma's `where` clause — the same
access-control-before-retrieval pattern as `demo04_preprocessing_tools` §6.6, but enforced by the
store itself rather than a Python filter over an in-memory list.

In [9]:
import dspy

class ChromaRetriever(dspy.Module):
    """A DSPy retriever backed by a Chroma collection."""
    def __init__(self, collection, embed_fn, k=5):
        self.collection, self.embed_fn, self.k = collection, embed_fn, k

    def forward(self, query, access_type="public"):
        res = self.collection.query(
            query_embeddings=[self.embed_fn(query).tolist()],
            n_results=self.k,
            where={"access": access_type},
        )
        return dspy.Prediction(passages=res["documents"][0])

retriever = ChromaRetriever(col, embed_query, k=3)
print("retriever ready over", col.count(), "chunks")

retriever ready over 7 chunks


## Example retrieval

The full pipeline in one query — a fact the extraction/cleaning/chunking steps above kept intact
inside its heading path. Then the same query re-run with `access_type="confidential"`, to show the
`where` clause actually scoping what comes back.

In [10]:
query = "What is the penalty for late submission?"
result = retriever(query)
print(f"query: {query!r}\n")
print("ChromaRetriever passages:")
for p in result.passages:
    print(" -", p.replace(chr(10), " ")[:100])

hits = col.query(query_embeddings=[embed_query(query).tolist()], n_results=3,
                where={"access": "public"})
print("\nsame hits, with page/section metadata for citation:")
for doc, meta, dist in zip(hits["documents"][0], hits["metadatas"][0], hits["distances"][0]):
    print(f"  [dist={dist:.3f}] (p{meta['page']}) [{meta['section']}]  {doc.replace(chr(10), ' ')[:70]}")

# access_type scopes retrieval before the LLM ever sees a passage
salary_q = "What is the base salary for senior lecturers?"
print(f"\nquery: {salary_q!r}")
print("public access      ->", retriever(salary_q, access_type="public").passages[0].replace(chr(10), " ")[:90])
print("confidential access ->", retriever(salary_q, access_type="confidential").passages[0].replace(chr(10), " "))

query: 'What is the penalty for late submission?'

ChromaRetriever passages:
 - ### **Late submission** Assignments submitted after the deadline are accepted for up to five days. E
 - ### **Grading** The passing grade for every course is 60 out of 100. A final grade below 60 requires
 - ## **Library Fees** The following fines apply to overdue library materials. |**Item type**|**Fine pe

same hits, with page/section metadata for citation:
  [dist=0.286] (p1) [**Course Policy** > **Assignments** > **Late submission**]  ### **Late submission** Assignments submitted after the deadline are a
  [dist=0.684] (p1) [**Course Policy** > **Assignments** > **Grading**]  ### **Grading** The passing grade for every course is 60 out of 100. A
  [dist=0.721] (p2) [**Campus Policies** > **Library Fees**]  ## **Library Fees** The following fines apply to overdue library mater

query: 'What is the base salary for senior lecturers?'
public access      -> ### **Grading** The passing grade for every course 

## Takeaways

- **Page number is metadata, for free** — `page_chunks=True` on the extractor carries it all the
  way to the Chroma collection, alongside the heading path from chunking.
- **Enrich before you dedup.** Dedup and near-duplicate detection compare the *enriched* embed text,
  not the raw chunk — that's why enrichment runs before it in the pipeline order.
- **The retriever doesn't care where the vectors came from.** `ChromaRetriever` is the same class
  whether it sits over arXiv chunks (`demo04_vector_stores`) or this handbook PDF.